# Hotel Booking Analysis
Project 1: Data Cleaning, Pre-processing, and 10 Analysis Tasks


### Prep: Data Cleaning and Pre-processing
**Task:** Load the data, remove duplicates, fix the date column, handle missing values (company, agent, country, children).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('hotel_bookings.csv')

# Remove duplicates
df = df.drop_duplicates()

# Fix date column
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

# Handle missing values
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

# Sort months chronologically
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
df['arrival_date_month'] = pd.Categorical(df['arrival_date_month'], categories=months, ordered=True)

print("Cleaned data shape:", df.shape)
df.head()


### Q1: Confirmed Bookings per Month
**Task:** Plot the number of confirmed (not cancelled) bookings per month for the City Hotel and the Resort Hotel on one graph.


In [ ]:
confirmed = df[df['is_canceled'] == 0]
q1 = confirmed.groupby(['arrival_date_month', 'hotel'], observed=False).size().unstack()

q1.plot(kind='bar', figsize=(10, 5), width=0.8)
plt.title('Confirmed Bookings per Month')
plt.xlabel('Month')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.show()


### Q2: Confirmed vs Cancelled Bookings Month by Month
**Task:** For each hotel, compare confirmed bookings with cancelled bookings month by month.


In [ ]:
for hotel_name in ['City Hotel', 'Resort Hotel']:
    hotel_df = df[df['hotel'] == hotel_name]
    q2 = hotel_df.groupby(['arrival_date_month', 'is_canceled'], observed=False).size().unstack()
    q2.columns = ['Confirmed', 'Cancelled']
    
    q2.plot(kind='bar', figsize=(10, 4), color=['green', 'red'], width=0.8)
    plt.title(f'{hotel_name}: Confirmed vs Cancelled Bookings')
    plt.xlabel('Month')
    plt.ylabel('Bookings')
    plt.xticks(rotation=45)
    plt.show()


### Q3: Average ADR for Each Month (City Hotel)
**Task:** Find the average ADR (Average Daily Rate) for each month for the City Hotel and plot it.


In [ ]:
city_hotel = df[df['hotel'] == 'City Hotel']
adr_monthly = city_hotel.groupby('arrival_date_month', observed=False)['adr'].mean()

plt.figure(figsize=(10, 5))
plt.plot(adr_monthly.index, adr_monthly.values, marker='o', color='b', linewidth=2)
plt.title('City Hotel: Average Daily Rate (ADR) per Month')
plt.xlabel('Month')
plt.ylabel('Average ADR ($)')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


### Q4: Relationship Between Monthly Cancellations and Average ADR in City Hotel
**Task:** Is there a relationship between monthly cancellations and average ADR in the City Hotel? Find the correlation and show it as a heatmap.


In [ ]:
city_stats = city_hotel.groupby('arrival_date_month', observed=False).agg(
    Cancellations=('is_canceled', lambda x: (x == 1).sum()),
    Average_ADR=('adr', 'mean')
)

corr = city_stats.corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Correlation Heatmap: Cancellations vs Average ADR')
plt.show()


### Q5: Percentage Share of Each Market Segment
**Task:** Find the percentage share of each market segment for both hotels.


In [ ]:
market_share = pd.crosstab(df['market_segment'], df['hotel'], normalize='columns') * 100

print(market_share.round(2))

market_share.plot(kind='bar', figsize=(10, 5))
plt.title('Market Segment Share (%) for Both Hotels')
plt.xlabel('Market Segment')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.show()


### Q6: Percentage of Repeated vs New Guests in City Hotel
**Task:** What percentage of City Hotel guests are repeated guests and what percentage are new?


In [ ]:
repeat_share = city_hotel['is_repeated_guest'].value_counts(normalize=True) * 100
repeat_share.index = ['New Guests', 'Repeated Guests']

print(repeat_share.round(2))

plt.figure(figsize=(5, 5))
plt.pie(repeat_share, labels=repeat_share.index, autopct='%1.2f%%', colors=['#4285F4', '#FBBC05'], startangle=90)
plt.title('City Hotel: New vs Repeated Guests')
plt.show()


### Q7: Most Reserved Room Types in Each Hotel
**Task:** Which room types are reserved the most in each hotel? Show the share of each type as a bar chart.


In [ ]:
room_share = pd.crosstab(df['reserved_room_type'], df['hotel'], normalize='columns') * 100

print(room_share.round(2))

room_share.plot(kind='bar', figsize=(10, 5))
plt.title('Reserved Room Type Share (%) by Hotel')
plt.xlabel('Room Type')
plt.ylabel('Percentage (%)')
plt.show()


### Q8: Share of Each Customer Type Among Confirmed Bookings
**Task:** What is the share of each customer type among confirmed bookings? Show it as a pie chart.


In [ ]:
customer_share = confirmed['customer_type'].value_counts(normalize=True) * 100

print(customer_share.round(2))

plt.figure(figsize=(6, 6))
plt.pie(customer_share, labels=customer_share.index, autopct='%1.2f%%', startangle=140)
plt.title('Customer Type Share Among Confirmed Bookings')
plt.show()


### Q9: Percentage of All Reservations Cancelled vs Not Cancelled
**Task:** What percentage of all reservations were cancelled and what percentage were not? Show it as a pie chart.


In [ ]:
cancel_share = df['is_canceled'].value_counts(normalize=True) * 100
cancel_share.index = ['Not Cancelled', 'Cancelled']

print(cancel_share.round(2))

plt.figure(figsize=(5, 5))
plt.pie(cancel_share, labels=cancel_share.index, autopct='%1.2f%%', colors=['#34A853', '#EA4335'], startangle=90)
plt.title('Total Reservations: Cancelled vs Not Cancelled')
plt.show()
